# 02 - Identificação de Fundos REAG

Identifica fundos administrados/geridos pela REAG usando dados de cadastro da CVM.

**REAG**: Administradora investigada por fraude junto ao Banco Master

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from pathlib import Path
from src.processors.data_processor import DataProcessor
from config.settings import Config

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

## Carregar Cadastro de Fundos

In [ ]:
config = Config()
processor = DataProcessor(config)

# Ler cadastro mais recente
cadastro_files = sorted(config.RAW_DATA_DIR.glob('cadastro_*.csv'))
latest_cadastro = cadastro_files[-1] if cadastro_files else None

if latest_cadastro:
    print(f"📂 Lendo: {latest_cadastro.name}")
    df_cadastro = processor.read_cadastro(latest_cadastro)
    print(f"📊 Total de fundos no cadastro: {len(df_cadastro):,}")
else:
    print("⚠️ Nenhum arquivo de cadastro encontrado")

## Buscar REAG/CBSF nos Administradores

In [ ]:
# Buscar por nome contendo 'REAG' ou 'CBSF'
search_terms = ['REAG', 'CBSF', 'BANCO MASTER']

mask = df_cadastro['DENOM_SOCIAL'].str.contains('|'.join(search_terms), case=False, na=False)
if 'ADMIN' in df_cadastro.columns:
    mask |= df_cadastro['ADMIN'].str.contains('|'.join(search_terms), case=False, na=False)
if 'GESTOR' in df_cadastro.columns:
    mask |= df_cadastro['GESTOR'].str.contains('|'.join(search_terms), case=False, na=False)

df_reag_related = df_cadastro[mask].copy()

print(f"\n🎯 Fundos relacionados encontrados: {len(df_reag_related)}")
print("\n📋 Amostra:")
display(df_reag_related[['CNPJ_FUNDO', 'DENOM_SOCIAL', 'SIT']].head(20))

## Identificar CNPJs de Administradores/Gestores REAG

In [ ]:
# Analisar administradores únicos
if 'CNPJ_ADMIN' in df_cadastro.columns and 'ADMIN' in df_cadastro.columns:
    admin_counts = df_cadastro.groupby(['CNPJ_ADMIN', 'ADMIN']).size().reset_index(name='NUM_FUNDOS')
    admin_counts = admin_counts.sort_values('NUM_FUNDOS', ascending=False)
    
    # Filtrar REAG
    reag_admins = admin_counts[
        admin_counts['ADMIN'].str.contains('|'.join(search_terms), case=False, na=False)
    ]
    
    print("\n🏢 Administradores REAG/relacionados:")
    display(reag_admins)
    
    # Salvar CNPJs para uso posterior
    reag_admin_cnpjs = reag_admins['CNPJ_ADMIN'].tolist()
    print(f"\n📝 CNPJs de administradores REAG: {len(reag_admin_cnpjs)}")
    print(reag_admin_cnpjs)
else:
    print("⚠️ Colunas de administrador não encontradas")
    reag_admin_cnpjs = []

## Obter Lista Completa de Fundos REAG

In [ ]:
# Filtrar fundos por CNPJ do administrador
if reag_admin_cnpjs:
    df_reag_funds = processor.filter_by_administrador(df_cadastro, reag_admin_cnpjs)
    
    print(f"\n💼 Total de fundos administrados pela REAG: {len(df_reag_funds)}")
    
    # Análise por situação
    if 'SIT' in df_reag_funds.columns:
        print("\n📊 Distribuição por situação:")
        display(df_reag_funds['SIT'].value_counts())
    
    # Fundos ativos
    df_reag_active = df_reag_funds[df_reag_funds['SIT'] == 'EM FUNCIONAMENTO NORMAL'].copy()
    print(f"\n✅ Fundos em funcionamento normal: {len(df_reag_active)}")
    
    # Salvar lista de CNPJs
    reag_fund_cnpjs = df_reag_funds['CNPJ_FUNDO'].unique().tolist()
    
    # Exportar para CSV
    output_path = config.PROCESSED_DATA_DIR / 'reag_fund_list.csv'
    df_reag_funds[['CNPJ_FUNDO', 'DENOM_SOCIAL', 'SIT', 'CNPJ_ADMIN']].to_csv(
        output_path, 
        index=False
    )
    print(f"\n💾 Lista salva em: {output_path}")
else:
    print("⚠️ Nenhum CNPJ de administrador REAG identificado")
    reag_fund_cnpjs = []

## Resumo

In [ ]:
print("\n" + "="*60)
print("📊 RESUMO DA IDENTIFICAÇÃO")
print("="*60)
print(f"Administradores REAG identificados: {len(reag_admin_cnpjs)}")
print(f"Fundos REAG identificados: {len(reag_fund_cnpjs)}")
print(f"Fundos ativos: {len(df_reag_active) if 'df_reag_active' in locals() else 0}")
print(f"\n📂 Lista exportada: {config.PROCESSED_DATA_DIR / 'reag_fund_list.csv'}")
print("\n✅ Identificação concluída! Próximo passo: 03_flow_analysis.ipynb")